In [1]:
%pip install scikit-learn matplotlib numpy torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Post Training Quantization-Dynamic

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
X, y=make_moons(n_samples=5000,noise=0.5,random_state=42)

In [5]:
X

array([[ 0.64527536,  1.38251014],
       [ 0.14514823, -0.32157033],
       [ 0.11945131,  0.41631146],
       ...,
       [ 0.6360473 ,  0.66530771],
       [ 1.61542641, -0.24249711],
       [ 0.10599548,  1.0899585 ]], shape=(5000, 2))

In [6]:
X.shape

(5000, 2)

In [7]:
X=StandardScaler().fit_transform(X)

In [8]:
X=torch.tensor(X,dtype=torch.float32)

In [9]:
y=torch.tensor(y.reshape(-1,1),dtype=torch.float32)

In [10]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
class BigMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(2,128)
        self.fc2=nn.Linear(128,64)
        self.fc3=nn.Linear(64,64)
        self.fc4=nn.Linear(64,32)
        self.fc5=nn.Linear(32,16)
        self.fc6=nn.Linear(16,8)
        self.fc7=nn.Linear(8,1)
    
    def forward(self,x):
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=F.relu(self.fc3(x))
        x=F.relu(self.fc4(x))
        x=F.relu(self.fc5(x))
        x=F.relu(self.fc6(x))
        return torch.sigmoid(self.fc7(x))

In [12]:
model_fp32=BigMLP()

In [13]:
model_fp32.parameters()

<generator object Module.parameters at 0x000001B0F9D07300>

In [14]:
optimizer=torch.optim.Adam(model_fp32.parameters(),lr=0.01)

In [15]:
loss_fn=nn.BCELoss()

In [16]:
for epoch in range(1500):
    model_fp32.train()
    optimizer.zero_grad()
    out=model_fp32(X_train)
    loss=loss_fn(out,y_train)
    loss.backward()
    optimizer.step()
    if epoch %500==0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 0.6937
Epoch 500 | Loss: 0.3819
Epoch 1000 | Loss: 0.3396
Epoch 1500 | Loss: 0.2977
Epoch 2000 | Loss: 0.2546


In [17]:
def accuracy(model,X,y):
    model.eval()
    with torch.no_grad():
        preds=model(X)
        preds=(preds>0.5).float()
        return (preds==y).float().mean().item()
    

In [18]:
accuracy(model_fp32,X_test,y_test)

0.7900000214576721

In [19]:
#Quantization Techniques
from torch.quantization import quantize_dynamic

#dynamixc quantization
model_int8=quantize_dynamic(
    model_fp32,#model name
    {nn.Linear},#which layer in models to quantize
    dtype=torch.qint8
)


C:\Users\ffmou\AppData\Local\Temp\ipykernel_15780\787055474.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8=quantize_dynamic(


In [20]:
print("INT8 Quantized accuracy:",accuracy(model_int8,X_test,y_test))

INT8 Quantized accuracy: 0.6880000233650208


In [21]:
import os
torch.save(model_fp32.state_dict(),"model_fp32.pt")
torch.save(model_int8.state_dict(),"model_dynamic_int8.pt")

In [22]:
print("FP32 model size(MB): ",os.path.getsize("model_fp32.pt")/1e6)
print("INT8 model size(MB):  ",os.path.getsize("model_dynamic_int8.pt")/1e6)

FP32 model size(MB):  0.067317
INT8 model size(MB):   0.026373


In [23]:
#quantize manually

def quantize_tensor(t,num_bits=8):
    qmin=-2**(num_bits-1)
    qmax=2**(num_bits-1)-1
    rmin=t.min()
    rmax=t.max()
    scale=(rmax-rmin)/float(qmax-qmin+1e-8)
    zp=torch.round(-rmin/scale).to(torch.int8)
    q_t=torch.clamp(torch.round(t/scale)+zp,qmin,qmax).to(torch.int8)
    return q_t,scale,zp

In [24]:
def dequantize_tensor(q_t,scale,zp):
    return (q_t.float()-zp)*scale

In [25]:
new_model=BigMLP()

In [26]:
for name,param in new_model.named_parameters():
    print(name)
    print(param.shape)

fc1.weight
torch.Size([128, 2])
fc1.bias
torch.Size([128])
fc2.weight
torch.Size([64, 128])
fc2.bias
torch.Size([64])
fc3.weight
torch.Size([64, 64])
fc3.bias
torch.Size([64])
fc4.weight
torch.Size([32, 64])
fc4.bias
torch.Size([32])
fc5.weight
torch.Size([16, 32])
fc5.bias
torch.Size([16])
fc6.weight
torch.Size([8, 16])
fc6.bias
torch.Size([8])
fc7.weight
torch.Size([1, 8])
fc7.bias
torch.Size([1])


In [27]:
with torch.no_grad():
    for (name_fp,parm_fp),(name_q,param_q) in zip(model_fp32.named_parameters(),new_model.named_parameters()):
        q_param,scale,zp=quantize_tensor(parm_fp.data)
        dq_param=dequantize_tensor(q_param,scale,zp)
        param_q.data.copy_(dq_param)

In [28]:
print("INT8 Accuracy: ",accuracy(new_model,X_test,y_test))

INT8 Accuracy:  0.6349999904632568


In [29]:
X, y=make_moons(n_samples=500,noise=0.2,random_state=42)

In [30]:
X=torch.tensor(X,dtype=torch.float32)
y=torch.tensor(y.reshape(-1,1),dtype=torch.float32)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [31]:
#PTQ Static
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))

model_fp32 = MLP()

optimizer = torch.optim.Adam(model_fp32.parameters(), lr=0.01)

loss_fn = nn.BCELoss()

for epoch in range(2000):
    model_fp32.train()
    optimizer.zero_grad()
    out = model_fp32(X_train)
    loss = loss_fn(out, y_train)
    loss.backward()
    optimizer.step()

  # Accuracy
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X)
        preds = (preds > 0.5).float()
        return (preds == y).float().mean().item()

print("\nFP32 Accuracy:", accuracy(model_fp32, X_test, y_test))


FP32 Accuracy: 0.9599999785423279


In [32]:

# Calibration: Get activation ranges
def get_activation_min_max(model, X_sample):
    act_ranges = {}
    hooks = []

    def register_hook(name):
        def hook_fn(module, input, output):
            act_min = output.min().item()
            act_max = output.max().item()
            act_ranges[name] = (act_min, act_max)
        return hook_fn

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(register_hook(name)))

    with torch.no_grad():
        model.eval()
        model(X_sample)

    for h in hooks:
        h.remove()

    return act_ranges

In [33]:
X_calib = X_train[:100]
     

act_ranges = get_activation_min_max(model_fp32, X_calib)
     

act_ranges
     

{'fc1': (-10.870328903198242, 8.52166748046875),
 'fc2': (-18.10552215576172, 31.335344314575195),
 'fc3': (-27.301790237426758, 167.2536163330078)}

In [34]:
class QuantizedMLP(nn.Module):
    def __init__(self, fp_model, act_ranges):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

        # Quantize weights
        with torch.no_grad():
            for (name_fp, param_fp), (name_q, param_q) in zip(fp_model.named_parameters(), self.named_parameters()):
                q_param, scale, zp = quantize_tensor(param_fp.data)
                dq_param = dequantize_tensor(q_param, scale, zp)
                param_q.data.copy_(dq_param)

        # Store activation scales
        self.act_scales = {}
        for name, (amin, amax) in act_ranges.items():
            scale = (amax - amin) / 255.0
            zp = round(-amin / scale) if scale > 0 else 0
            self.act_scales[name] = (scale, zp)

    def quant_act(self, x, layer_name):
        scale, zp = self.act_scales[layer_name]
        q_x = torch.clamp(torch.round(x / scale) + zp, 0, 255).to(torch.uint8)
        dq_x = (q_x.float() - zp) * scale
        return dq_x

    def forward(self, x):
        x = F.relu(self.quant_act(self.fc1(x), 'fc1'))
        x = F.relu(self.quant_act(self.fc2(x), 'fc2'))
        return torch.sigmoid(self.fc3(x))
     


In [35]:

model_static_ptq = QuantizedMLP(model_fp32, act_ranges)
print("STATIC PTQ Accuracy:", accuracy(model_static_ptq, X_test, y_test))

STATIC PTQ Accuracy: 0.7099999785423279
